# Study — the camera models (`mppp_studies.calibration`)

How the promoted camera models of `mppp cmods` are made: the Navcam joint over many blocks (the focal-length and
principal-point laws in temperature, the rig between the eyes, the Hazcam laws), the Mastcam-Z joint (the lens
law in focus count per zoom, the backlash states, the rig), the rover-deck adjustment that ties the Mastcam-Z rig
to the Navcam one, and the promotion.  The study reads finished blocks (`mppp align`, the `error_input/` folders) and
never changes them; the pipeline reads only the promoted files.

The runs are long (hours over the consensus blocks) and are started from the command line — the freeze driver
(`scripts/windows/freeze.bat`, `mppp_studies.calibration.freeze`) runs the whole chain unattended.  This notebook
shows the commands, and reads and plots their results.  `docs/methods.md` describes the models.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows/conda: duplicate Intel OpenMP guard; before torch
import json
import logging
from pathlib import Path
import mppp
logging.basicConfig(level=logging.INFO, format="%(message)s")   # the progress lines
print("MPPP", mppp.__version__, "from", Path(mppp.__file__).parent)
import mppp_studies
from mppp.calib import models
print("studies from", Path(mppp_studies.__file__).parent)

## 1  Parameters

In [ ]:
ROOT = Path(os.environ.get("MPPP_SCAPES", "/scapes"))     # the blocks root (the freeze blocks on the PC)
ANALYSIS = ROOT / "camera_analysis"                        # where the joints write (calib ... --out)
ZOOMS = (34, 63, 79, 110)
RUN = False                                                # True: run the joints from here (hours); False: read the results

## 2  The promoted models in use
`mppp cmods` — the files the pipeline reads (`MPPP_CMODS`, else the package data): per eye a `.json` with the
law and its covariance; the rig; the Mastcam-Z lens model.  `describe` prints what is promoted and from where.

In [ ]:
print("\n".join(models.describe()))
print(json.dumps(models.status(), indent=1, default=str)[:1500])

## 3  The Navcam joint — `mppp calib navcam`
All the consensus blocks' Navcam frames in one adjustment: each eye's fx, fy, cx, cy on a line in the camera
temperature (with the knee), the distortion shared, the rig between the eyes (rotation, the "deck" translation),
every frame's pose free.  The blocks' names come from `sites.json` (`ecam_consensus`).

In [ ]:
cmd = f"python -m mppp calib navcam --root {ROOT} --group ecam_consensus"
print(cmd)
if RUN:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "mppp", "calib", "navcam", "--root", str(ROOT), "--group", "ecam_consensus"], check=True)
runs = sorted(ANALYSIS.glob("navcam_joint_*")) if ANALYSIS.is_dir() else []
print("results:", [r.name for r in runs][-5:] or "none yet")

In [ ]:
from IPython.display import Image as NbImage, display
if runs:
    res = runs[-1]
    st = json.loads((res / "navcam_joint.json").read_text(encoding="utf-8")) if (res / "navcam_joint.json").is_file() else {}
    for eye, e in (st.get("eyes") or {}).items():
        print(eye, {k: e[k] for k in e if k in ("fx", "fy", "cx", "cy", "slope_fx_ppm_per_K", "knee_degC")})
    for png in sorted(res.glob("*.png"))[:4]:
        display(NbImage(filename=str(png)))

## 4  The Mastcam-Z joint — `mppp calib zcam --zoom 34`
Per zoom: the lens law in the focus motor count (f, fy/fx, cx, cy on a line per backlash state, the distortion on
the line), a pooled diagnostic pass per sequence, the rig to the Navcam block, the per-block aspect gate, the
weak-geometry sequences.  The backlash states come from the labels and the blocks (`mppp zcam-states`, the found
list `mars2020_mastcam-z_low_backlash_observations.csv`).

In [ ]:
for z in ZOOMS:
    print(f"python -m mppp calib zcam --zoom {z} --root {ROOT} --group zcam{z}_consensus --zcam-pool sequence")
zres = {z: sorted(ANALYSIS.glob(f"zcam{z}_joint_*")) for z in ZOOMS} if ANALYSIS.is_dir() else {}
for z, rs in zres.items():
    if rs:
        j = rs[-1] / "zcam_joint.json"
        d = json.loads(j.read_text(encoding="utf-8")) if j.is_file() else {}
        print(z, rs[-1].name, "promotable:", d.get("promotable"), "| failed checks:", d.get("failed_checks"))
        for png in sorted(rs[-1].glob("zcam_efl_vs_focus*.png"))[:1] + sorted(rs[-1].glob("zcam_rig*.png"))[:1]:
            display(NbImage(filename=str(png)))

## 5  The rover deck and the mast head
`python -m mppp_studies.calibration.deck WORK --zcam-rig full --rig full`: the deck fiducials (seven surveyed
targets) in the bundle adjustment of one block, which ties the Mastcam-Z rig to the Navcam rig and gives the mast
model (`claude/mast-head-2026-11-08-...`).  `zcam_mast_head` draws the cameras on the mast head from the promoted
rigs; `deck_rig_trend` the rig drift over the mission from the deck pairs; `fiducial_check` reads the fiducials
only.

In [ ]:
from mppp_studies.paths import outputs_dir
print("python -m mppp_studies.calibration.zcam_mast_head --root", ROOT, "--chain <the deck joint's chain json>")
print("python -m mppp_studies.calibration.deck_rig_trend --root", ROOT)
print("python -m mppp_studies.calibration.fiducial_check --root", ROOT)
for png in sorted(outputs_dir().glob("*mast_head*/*.png"))[:2] + sorted(outputs_dir().glob("*deck_rig_trend*/*.png"))[:1]:
    display(NbImage(filename=str(png)))

## 6  Promotion — `python -m mppp_studies.calibration.promote_cmods RESULT_DIR --note "..."`
A joint that passes its checks writes candidate files; `promote_cmods` copies them into the cmods folder with a
line in `CHANGES.md`, and `mppp cmods` shows them.  `calib zcam --promote-waive CHECK` promotes past a named failed
check (the 0.96 freeze waives the per-block aspect gate for 63, 79 and 110 mm: unobservable there).

In [ ]:
print("python -m mppp_studies.calibration.promote_cmods", ANALYSIS / "navcam_joint_YYYYMMDD", "--note 'the 0.96 freeze'")
print("python -m mppp cmods")